# Question Answering

**RAG 流程小结**：本 notebook 是 RAG 流程最后一步——**问答（Question Answering）**。前面的检索器只负责"找到相关文档"，
这里用 `RetrievalQA` 这个链（Chain）把"检索"和"生成回答"串起来：先用 retriever 找到相关文档，
再把这些文档拼进 prompt 交给 LLM 生成最终答案。同时会对比 stuff / map_reduce / refine 几种不同的"把多个文档喂给 LLM"的策略。

In [ ]:
import os
import openai
import sys
sys.path.append('../..')

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

openai.api_key  = os.environ['OPENAI_API_KEY']

In [ ]:
# 课程录制时 OpenAI 曾经历一次模型改版（gpt-3.5-turbo-0301 -> gpt-3.5-turbo），
# 这段逻辑是根据当前日期自动选用合适的模型名，避免用到已经过时/下线的旧模型版本号
import datetime
current_date = datetime.datetime.now().date()
if current_date < datetime.date(2023, 9, 2):
    llm_name = "gpt-3.5-turbo-0301"
else:
    llm_name = "gpt-3.5-turbo"
print(llm_name)

In [ ]:
# 正确路径：langchain_community.vectorstores.Chroma + langchain_openai.OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
persist_directory = 'docs/chroma/'
# TODO: 请在此处补全代码
# 1. 构造 embedding = OpenAIEmbeddings()
# 2. 用 persist_directory + embedding_function 打开 Chroma 向量库，得到 vectordb
embedding = None  # TODO
vectordb = None  # TODO

In [ ]:
# 确认向量库里已经有数据（应该和 03 节存入的块数一致）
print(vectordb._collection.count())

In [ ]:
# 先单独验证一下检索这一步是否正常工作，再往下接入 LLM 做完整问答
question = "What are major topics for this class?"
docs = vectordb.similarity_search(question,k=3)
len(docs)

In [ ]:
# ChatOpenAI 正确路径是 langchain_openai（旧版 langchain_community.chat_models 已不存在）
from langchain_openai import ChatOpenAI
# TODO: 请在此处补全代码（构造 llm = ChatOpenAI(model_name=llm_name, temperature=0)）
llm = None  # TODO

In [ ]:
# RetrievalQA 正确路径是 langchain_classic.chains（旧版 langchain_community.chains 里没有这个类）
from langchain_classic.chains import RetrievalQA

In [ ]:
# TODO: 请在此处补全代码
# 用 RetrievalQA.from_chain_type(llm, retriever=vectordb.as_retriever()) 构造 qa_chain（默认 "stuff" 策略）
qa_chain = None  # TODO

In [ ]:
# 【版本提示】qa_chain(...) 这种直接调用 Chain 对象的写法（对应 Chain.__call__）在新版 langchain-core 里
# 已被标记为 deprecated（推荐用 qa_chain.invoke(...) 代替），但目前仍然可以正常运行，只是会打印一条 deprecation warning，
# 属于"能跑但不推荐"的情况，这里保留课程原写法不做修改
result = qa_chain({"query": question})

In [ ]:
# RetrievalQA 返回的是一个 dict，"result" 键对应 LLM 生成的最终回答文本
result["result"]

In [ ]:
# 自定义 prompt 模板：Prompt Engineering 是 RAG 效果的关键一环。
# {context} 会被自动替换成检索到的文档内容，{question} 替换成用户问题。
# 模板里明确要求"不知道就说不知道，别编答案"（减少幻觉）、限制回答长度（三句话以内），
# 并要求固定结尾语（"thanks for asking!"，方便验证 prompt 真的生效了）
from langchain_core.prompts import PromptTemplate

# Build prompt
template = """Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer. Use three sentences maximum. Keep the answer as concise as possible. Always say "thanks for asking!" at the end of the answer.
{context}
Question: {question}
Helpful Answer:"""
QA_CHAIN_PROMPT = PromptTemplate.from_template(template)


In [ ]:
# TODO: 请在此处补全代码
# 用 RetrievalQA.from_chain_type(llm, retriever=vectordb.as_retriever(),
#   return_source_documents=True, chain_type_kwargs={"prompt": QA_CHAIN_PROMPT}) 构造 qa_chain
qa_chain = None  # TODO

In [ ]:
question = "Is probability a class topic?"

In [ ]:
# TODO: 请在此处补全代码（调用 qa_chain({"query": question}) 得到 result）
result = None  # TODO

In [ ]:
# 检查回答是否遵守了自定义 prompt 的要求（应该以 "thanks for asking!" 结尾）
result["result"]

In [ ]:
# 因为传了 return_source_documents=True，可以看到回答具体是依据哪个 Document 生成的，便于溯源核查
result["source_documents"][0]

In [ ]:
# chain_type="map_reduce"：先对每个文档块分别生成小结（map），再汇总成最终答案（reduce）
# TODO: 请在此处补全代码（构造 qa_chain_mr，chain_type="map_reduce"）
qa_chain_mr = None  # TODO

In [ ]:
result = qa_chain_mr({"query": question})

In [ ]:
# map_reduce 因为丢失了跨文档块的上下文，回答质量有时反而不如简单的 stuff 策略，可以和上面的结果对比感受一下
result["result"]

In [ ]:
# 如果想用 LangSmith 追踪 map_reduce/refine 每一步的中间调用细节，可以取消下面的注释并填入真实 API Key
#import os
#os.environ["LANGCHAIN_TRACING_V2"] = "true"
#os.environ["LANGCHAIN_ENDPOINT"] = "https://api.langchain.plus"
#os.environ["LANGCHAIN_API_KEY"] = "..." # replace dots with your api key

In [ ]:
# 重新跑一遍 map_reduce（如果上面打开了 LangSmith 追踪，这里就能在 LangSmith 后台看到 map/reduce 每一步的详细过程）
qa_chain_mr = RetrievalQA.from_chain_type(
    llm,
    retriever=vectordb.as_retriever(),
    chain_type="map_reduce"
)
result = qa_chain_mr({"query": question})
result["result"]

In [ ]:
# chain_type="refine"：迭代式地用每个新文档块去修正/精炼上一轮的答案
# TODO: 请在此处补全代码
# 1. 构造 qa_chain_mr，chain_type="refine"
# 2. 调用 qa_chain_mr({"query": question}) 得到 result 并打印 result["result"]
qa_chain_mr = None  # TODO
result = None  # TODO

In [ ]:
# 回到最基础的 stuff 策略，准备演示 RetrievalQA 缺乏对话记忆的问题
# TODO: 请在此处补全代码（构造 qa_chain，默认 stuff 策略）
qa_chain = None  # TODO

In [ ]:
question = "Is probability a class topic?"
result = qa_chain({"query": question})
result["result"]

In [ ]:
# 这里的 "those prerequisites" 指代的是上一个问题里提到的内容，但 RetrievalQA 不记得上一轮对话，
# 检索时只会拿 "why are those prerequesites needed?" 这句话本身去做相似度检索，很可能检索不到真正相关的内容，
# 导致回答答非所问——这正是 06 节要解决的"对话记忆"问题
question = "why are those prerequesites needed?"
result = qa_chain({"query": question})
result["result"]